# Файлы и форматы данных

Программа может читать данные из файла и сохранять результат, чтобы он не пропал после завершения работы. В этом уроке научимся работать с текстовыми файлами, путями, CSV и JSON.

> Ячейки создают учебные файлы в текущей папке блокнота. Запускай их по порядку.

## Карта урока

1. Открываем файл с помощью `with open(...)`
2. Читаем и записываем текст
3. Разбираемся с путями и кодировками
4. Читаем и записываем CSV
5. Читаем и записываем JSON
6. Практика: сохранить список дел


## 1. Открываем файл с помощью `with open(...)`

Чтобы читать или записывать файл, его сначала нужно открыть. Конструкция `with` закрывает файл автоматически, даже если во время работы возникнет ошибка.


![Открытие файла, чтение и автоматическое закрытие](assets/with_open.png)


In [ ]:
from pathlib import Path

notes_path = Path("lesson_notes.txt")

with open(notes_path, "w", encoding="utf-8") as file:
    file.write("Первая строка заметки.\n")
    file.write("Вторая строка заметки.\n")

Второй аргумент `"w"` означает режим записи. Если файла ещё нет, Python создаст его. Если файл уже есть, его прежнее содержимое будет заменено.

После блока `with` файл закрыт. Не нужно отдельно вызывать `file.close()`.

## 2. Читаем текст из файла

Режим `"r"` открывает файл для чтения. Метод `read()` читает всё содержимое.


In [ ]:
with open(notes_path, "r", encoding="utf-8") as file:
    text = file.read()

print(text)

Чтобы прочитать файл построчно, можно пройтись по нему циклом `for`. Такой способ удобен для больших файлов.


In [ ]:
with open(notes_path, "r", encoding="utf-8") as file:
    for line in file:
        print(line.rstrip())

В конце каждой строки в файле часто есть символ переноса строки `\n`. Метод `rstrip()` убирает его с конца вывода.

### Добавляем текст

Режим `"a"` добавляет данные в конец существующего файла. Если файла нет, он будет создан.


In [ ]:
with open(notes_path, "a", encoding="utf-8") as file:
    file.write("Добавили ещё одну строку.\n")

with open(notes_path, "r", encoding="utf-8") as file:
    print(file.read())

Основные режимы открытия:

| Режим | Что делает |
|---|---|
| `"r"` | Читает существующий файл |
| `"w"` | Записывает с начала; старое содержимое заменяется |
| `"a"` | Добавляет в конец |
| `"x"` | Создаёт новый файл; выдаёт ошибку, если такой файл уже есть |

Для обычного текста указывай `encoding="utf-8"`. Так Python одинаково прочитает кириллицу на разных компьютерах.


## 3. Пути к файлам

**Путь** указывает, где файл находится. Относительный путь считается от текущей рабочей папки программы. `Path.cwd()` показывает эту папку, а оператор `/` помогает соединять части пути.


In [ ]:
from pathlib import Path

current_folder = Path.cwd()
data_folder = current_folder / "lesson_data"
notes_file = data_folder / "notes.txt"

print("Текущая папка:", current_folder)
print("Путь к файлу:", notes_file)

Создадим отдельную папку для учебных данных. Параметр `parents=True` при необходимости создаёт и родительские папки, а `exist_ok=True` не вызывает ошибку, если папка уже существует.


In [ ]:
data_folder.mkdir(parents=True, exist_ok=True)

with open(notes_file, "w", encoding="utf-8") as file:
    file.write("Файл лежит в папке lesson_data.\n")

print(notes_file.exists())

Если читаемый файл не найден, Python поднимет `FileNotFoundError`. Обработай её, если отсутствие файла — ожидаемая ситуация.


In [ ]:
missing_file = data_folder / "not_created.txt"

try:
    with open(missing_file, "r", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("Файл пока не создан.")

`pathlib` также умеет читать и записывать текст одной командой. Для первых упражнений используем `with open(...)`, чтобы привыкнуть к явным режимам и автоматическому закрытию.


In [ ]:
quick_note = data_folder / "quick_note.txt"
quick_note.write_text("Короткая запись через pathlib.\n", encoding="utf-8")
print(quick_note.read_text(encoding="utf-8"))

## 4. Формат CSV

CSV хранит таблицу: одна строка файла соответствует одной строке таблицы. Для работы с ним в Python есть встроенный модуль `csv`.

Передадим `newline=""` при открытии CSV-файла. Это помогает корректно записывать строки на разных операционных системах.


In [ ]:
import csv

students_path = data_folder / "students.csv"
students = [
    {"name": "Аня", "score": 5},
    {"name": "Борис", "score": 4},
    {"name": "Вера", "score": 5},
]

with open(students_path, "w", newline="", encoding="utf-8") as file:
    fieldnames = ["name", "score"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(students)

`DictReader` читает каждую строку как словарь, где ключи — это заголовки колонок.


In [ ]:
with open(students_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row["name"], row["score"])

Значения, прочитанные из CSV, представлены строками. Если поле нужно использовать как число, преобразуй его явно.


In [ ]:
with open(students_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        score = int(row["score"])
        print(f"{row['name']}: {score + 1}")

CSV подходит для простых таблиц. Если данные вложены друг в друга или у разных записей разные поля, удобнее может оказаться JSON.

## 5. Формат JSON

JSON хранит структуры, похожие на словари, списки, строки, числа и логические значения Python. Для преобразования между Python и JSON используется модуль `json`.


In [ ]:
import json

profile = {
    "name": "Аня",
    "city": "Казань",
    "hobbies": ["шахматы", "робототехника"],
    "active": True,
}

profile_path = data_folder / "profile.json"
with open(profile_path, "w", encoding="utf-8") as file:
    json.dump(profile, file, ensure_ascii=False, indent=2)

Параметр `ensure_ascii=False` оставляет кириллицу читаемой в файле. `indent=2` добавляет отступы, чтобы JSON было удобно смотреть.

Прочитаем JSON обратно. Метод `json.load()` читает файл и превращает данные в словарь Python.


In [ ]:
with open(profile_path, "r", encoding="utf-8") as file:
    loaded_profile = json.load(file)

print(loaded_profile)
print(loaded_profile["hobbies"][0])

Запомни разницу:

- `json.dump(data, file)` записывает объект Python в открытый файл;
- `json.load(file)` читает объект Python из открытого файла.

Для строк есть `json.dumps(data)` и `json.loads(text)` — они не работают с файловым объектом.


In [ ]:
data_as_text = json.dumps(profile, ensure_ascii=False)
profile_copy = json.loads(data_as_text)

print(data_as_text)
print(profile_copy["city"])

JSON хранит данные, а не исполняемый код. В JSON нет прямого аналога кортежей и множеств: при сохранении используй поддерживаемые типы, например список.

## 6. Мини-проект: список дел

Сделаем список задач и сохраним его в JSON-файл. В каждой задаче будет текст и отметка, выполнена ли она.


In [ ]:
tasks_path = data_folder / "tasks.json"
tasks = [
    {"title": "Прочитать урок", "done": True},
    {"title": "Решить задачу", "done": False},
]

with open(tasks_path, "w", encoding="utf-8") as file:
    json.dump(tasks, file, ensure_ascii=False, indent=2)

Загрузим задачи и покажем их в удобном виде.


In [ ]:
with open(tasks_path, "r", encoding="utf-8") as file:
    saved_tasks = json.load(file)

for task in saved_tasks:
    mark = "✓" if task["done"] else " "
    print(f"[{mark}] {task['title']}")

### Практика

1. Добавь в список дел новую задачу.
2. Сохрани обновлённый список в `tasks.json`.
3. Прочитай файл ещё раз и выведи все задачи.
4. Добавь проверку на отсутствие файла при чтении.


In [ ]:
# Продолжи мини-проект здесь


## Домашние задания

### Базовый уровень

1. Запиши три любимые книги в текстовый файл и прочитай его построчно.
2. Добавь ещё одну строку в файл с помощью режима `"a"`.
3. Создай CSV-файл с названиями товаров и ценами. Прочитай таблицу и выведи товары дороже заданной суммы.

### Средний уровень

4. Сохрани словарь с настройками игры в JSON. Прочитай его и измени одно значение.
5. Сохрани список словарей об учениках в CSV и прочитай его с помощью `DictReader`.
6. Напиши программу, которая добавляет введённую задачу в список `tasks.json`. Если файла ещё нет, создай пустой список.

### Задание со звёздочкой ★

Сделай небольшой каталог книг. Для каждой книги храни название, автора и год. Сохраняй каталог в JSON, а затем добавь поиск по автору.


## Короткое повторение

- Используй `with open(path, mode, encoding="utf-8") as file:` — файл закроется автоматически.
- `"r"` читает, `"w"` перезаписывает, `"a"` добавляет в конец.
- Для путей удобно применять `pathlib.Path`.
- Для CSV используй `csv.DictReader` и `csv.DictWriter`.
- Для JSON используй `json.load()` и `json.dump()`.
- Числа из CSV приходят строками, поэтому преобразуй их явно.
- Обрабатывай ожидаемую ошибку `FileNotFoundError`, если файла может не быть.

### Проверь себя

1. Чем режим `"a"` отличается от `"w"`?
2. Зачем указывать `encoding="utf-8"`?
3. Какой объект возвращает `json.load()` для JSON-объекта?
4. Какой параметр CSV-ридера даёт строкам имена колонок?
